### Project Overview
This project explores abstractive summarization on the BillSum dataset using a custom encoder-decoder Transformer architecture. The goal is to understand how Transformer components such as tokenization, positional encoding, masking, encoder-decoder attention, and greedy decoding contribute to summarization.
## Research Question
Can a custom Transformer trained from scratch learn to summarize legal bills, and how does performance change as the training set size increases?

In [1]:
!pip install datasets transformers sentence-transformers faiss-cpu langchain evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=a2909e3b3c9e98d92deb8a802180e1857d5935452f10dbdae5c47f58f68ace80
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
#We decided to work with FiscalNote/billsum dataset from Hugging Face
dataset = load_dataset("FiscalNote/billsum")
#We decided to use pretrained Facebook/bart-base tokenizer which was pretrained on 50,265 tokens
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

data/ca_test-00000-of-00001.parquet:   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

...

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 18949
    })
    test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 1237
    })
})


## Dataset: BillSum

BillSum contains long legislative bill texts paired with human-written summaries. Each example contains:
- `text`: full bill text
- `summary`: reference summary
- `title`: bill title

This makes the task a sequence-to-sequence problem: the model receives bill text as input and generates a summary as output.

In [4]:
sample_train = dataset['train'][0].keys() #We extracted the train key from the dataset
print(sample_train)

dict_keys(['text', 'summary', 'title'])


## Tokenization and Preprocessing

Instead of using our own custom Tokenizer we use the BART tokenizer because it provides a strong pretrained subword vocabulary. However, for Model 1, only the tokenizer is reused — the Transformer model itself is trained from scratch.

The bill text is padded to a max length of 512 tokens, while the summary is padded to a max length of 128 tokens. This creates fixed-size inputs for batching.

In [5]:
def preprocess(data):
  source = tokenizer(data["text"], max_length = 512, truncation = True, padding = "max_length")
  target = tokenizer(data["summary"], max_length=128, truncation=True, padding="max_length")
  source["labels"] = target["input_ids"]
  return source

In [6]:
train_dataset = dataset["train"].select(range(100))
tokenized_dataset = train_dataset.map(preprocess)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [7]:
tokenized_dataset.set_format(type="torch")

In [8]:
print(tokenized_dataset[0].keys())

dict_keys(['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'])


In [9]:
print(tokenized_dataset[0]["input_ids"].shape)
print(tokenized_dataset[0]["labels"].shape)

torch.Size([512])
torch.Size([128])


In [10]:
from torch.utils.data import DataLoader
dataloader = DataLoader(tokenized_dataset, batch_size=8, shuffle=True)
val_dataset = dataset["test"].select(range(50))
val_tokenized = val_dataset.map(preprocess)
val_tokenized.set_format(type="torch")
val_dataloader = DataLoader(val_tokenized, batch_size=8)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

## Decoder Input and Target Shifting

For sequence generation, the decoder learns to predict the next token.

If the true summary is:

`<bos> The bill authorizes funding <eos>`

then:

`decoder_input = <bos> The bill authorizes funding`

`decoder_target = The bill authorizes funding <eos>`

This teaches the model next-token prediction during training.

The reason the decoder is shifted is because the position gives the decoder the cue as to what is coming next.

In [11]:
batch = next(iter(dataloader))

src = batch["input_ids"]
labels = batch["labels"]

decoder_input = labels[:, :-1]
decoder_target = labels[:, 1:]

## Padding and Causal Masks

Two types of masks are used:

* Padding masks which prevent the model from attending to `<pad>` tokens.
* Casual masks which prevent the decoder from seeing future tokens during training.

This is essential because the decoder should only use previous summary tokens when predicting the next one.

In [12]:
pad_token_id = tokenizer.pad_token_id

src_padding_mask = src == pad_token_id
tgt_padding_mask = decoder_input == pad_token_id
memory_key_padding_mask = src_padding_mask

In [13]:
import torch
def generate_subsequent_mask(size):
    return torch.triu(torch.ones(size, size), diagonal=1).bool()

tgt_mask = generate_subsequent_mask(decoder_input.size(1)).to(src.device)

In [14]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

## Positional Encoding

Transformers do not naturally understand token order. Positional encoding is a way to encode a word's position so that the transformer can understand the information. This allows the model to differentiate between the same words appearing in different positions.

In [15]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_length=1000, dropout=0.1):
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    #Create a long enough P
    P = torch.zeros((1, max_length, d_model))
    pos = torch.arange(0, max_length, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * -math.log(10000.0)/ d_model)
    P[:,:,0::2] = torch.sin(pos * div_term)
    P[:,:,1::2] = torch.cos(pos * div_term)
    self.register_buffer('P', P)
  def forward(self, X):
    X = X + self.P[:, :X.shape[1], :]
    return self.dropout(X)

## Model 1: Custom Encoder-Decoder Transformer

The custom model contains:
* source and target embedding layers
* sinusoidal positional encoding
* Transformer encoder-decoder layers
* final linear projection to the tokenizer vocabulary

The model outputs logits with shape:

[batch_size, target_sequence_length, vocabulary_size]

Each position predicts the next summary token.

In [16]:
import math
class CustomSeq2SeqTransformer(nn.Module):
  def __init__(self, no_enc_layers, no_dec_layers, emb_size, no_head, src_vcb_size, tgt_vcb_size, dim_feedforward, dropout):
    super().__init__()
    self.emb_size = emb_size
    self.src_embedding = nn.Embedding(src_vcb_size, embedding_dim=emb_size)
    self.trgt_embedding = nn.Embedding(tgt_vcb_size, embedding_dim=emb_size)
    self.positional_encoding = PositionalEncoding(d_model=emb_size, max_length=1024, dropout=dropout)
    self.transformer = nn.Transformer(
        d_model=emb_size,
        nhead=no_head,
        num_encoder_layers=no_enc_layers,
        num_decoder_layers=no_dec_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout,
        batch_first=True
    )
    self.feedforward = nn.Linear(emb_size,tgt_vcb_size)
  def forward(self, src, trg, src_mask, trg_mask, src_padding_mask, tgt_padding_mask, memory_key_padding_mask):
    source_emb = self.src_embedding(src) * math.sqrt(self.emb_size)
    target_emb = self.trgt_embedding(trg) * math.sqrt(self.emb_size)
    source_emb = self.positional_encoding(source_emb)
    target_emb = self.positional_encoding(target_emb)
    output = self.transformer(
        source_emb,
        target_emb,
        src_mask,
        trg_mask,
        None,
        src_padding_mask,
        tgt_padding_mask,
        memory_key_padding_mask
    )
    return self.feedforward(output)

In [17]:
#Test a forward pass
model = CustomSeq2SeqTransformer(
    no_enc_layers=2,
    no_dec_layers=2,
    emb_size=128,
    no_head=4,
    src_vcb_size=len(tokenizer),
    tgt_vcb_size=len(tokenizer),
    dim_feedforward=512,
    dropout=0.1
)

output = model(
    src,
    decoder_input,
    src_mask=None,
    trg_mask=tgt_mask,
    src_padding_mask=src_padding_mask,
    tgt_padding_mask=tgt_padding_mask,
    memory_key_padding_mask=memory_key_padding_mask
)

print(output.shape)

torch.Size([8, 127, 50265])


## Experiment 1: 100-Example

The first experiment trains the custom Transformer on 100 examples. The purpose is not to achieve high ROUGE, but to confirm that the model can learn and that training/validation loss decreases.

In [18]:
import time
import torch
import torch.nn as nn
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=pad_token_id)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

no_epochs = 30
best_vloss= 1_000_000.
model_transformer_path= ""
start_time = time.time()

pad_token_id = tokenizer.pad_token_id

for epoch in range(no_epochs):
    running_loss = 0.0
    epoch_start_time = time.time()
    # Here, we use enumerate(training_loader) instead of
    # iter(training_loader) so that we can track the batch
    # index and do some intra-epoch reporting
    #We initialize the model to training mode
    model.train()
    for i, data in enumerate(dataloader):
      src = data["input_ids"].to(device)
      labels = data["labels"].to(device)

      decoder_input = labels[:, :-1]
      decoder_target = labels[:, 1:]


      src_padding_mask = src == pad_token_id
      tgt_padding_mask = decoder_input == pad_token_id
      memory_key_padding_mask = src_padding_mask

      tgt_mask = generate_subsequent_mask(decoder_input.size(1)).to(device)

      optimizer.zero_grad()

      output = model( src, decoder_input, src_mask=None, trg_mask=tgt_mask, src_padding_mask=src_padding_mask,
                     tgt_padding_mask=tgt_padding_mask, memory_key_padding_mask=memory_key_padding_mask)
      vocab_size = output.shape[-1]
      loss = loss_fn(output.reshape(-1, vocab_size), decoder_target.reshape(-1))

      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
      optimizer.step()

      running_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
      for i, val_data in enumerate(val_dataloader):
        src = val_data["input_ids"].to(device)
        labels = val_data["labels"].to(device)

        decoder_input = labels[:, :-1]
        decoder_target = labels[:, 1:]


        src_padding_mask = src == pad_token_id
        tgt_padding_mask = decoder_input == pad_token_id
        memory_key_padding_mask = src_padding_mask

        tgt_mask = generate_subsequent_mask(decoder_input.size(1)).to(device)
        output = model(src,decoder_input,src_mask=None,trg_mask=tgt_mask,
                         src_padding_mask=src_padding_mask,tgt_padding_mask=tgt_padding_mask,
                         memory_key_padding_mask=memory_key_padding_mask)

        vocab_size = output.shape[-1]

        loss = loss_fn(output.reshape(-1, vocab_size),decoder_target.reshape(-1))

        val_loss += loss.item()


    avg_train_loss = running_loss / len(dataloader)
    avg_val_loss = val_loss / len(val_dataloader)
    if avg_val_loss < best_vloss:
      best_vloss = avg_val_loss
      torch.save(model.state_dict(), "best_custom_transformer.pt")
    epoch_end = time.time()

    print( f"Epoch {epoch+1}/{no_epochs} | ", f"Train Loss: {avg_train_loss:.4f} | ", f"Val Loss: {avg_val_loss:.4f} | ", f"Time: {epoch_end - epoch_start_time:.2f}s")

end_time = time.time()
print("Finished training. Time:", end_time - start_time, "seconds")



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1/30 |  Train Loss: 10.4793 |  Val Loss: 10.0227 |  Time: 1.53s
Epoch 2/30 |  Train Loss: 9.6749 |  Val Loss: 9.3871 |  Time: 0.47s
Epoch 3/30 |  Train Loss: 8.9426 |  Val Loss: 8.7629 |  Time: 0.47s
Epoch 4/30 |  Train Loss: 8.2289 |  Val Loss: 8.2060 |  Time: 0.44s
Epoch 5/30 |  Train Loss: 7.6142 |  Val Loss: 7.7749 |  Time: 0.48s
Epoch 6/30 |  Train Loss: 7.1484 |  Val Loss: 7.4762 |  Time: 0.50s
Epoch 7/30 |  Train Loss: 6.8145 |  Val Loss: 7.2960 |  Time: 0.48s
Epoch 8/30 |  Train Loss: 6.5999 |  Val Loss: 7.2060 |  Time: 1.40s
Epoch 9/30 |  Train Loss: 6.4686 |  Val Loss: 7.1364 |  Time: 0.45s
Epoch 10/30 |  Train Loss: 6.3677 |  Val Loss: 7.0755 |  Time: 0.45s
Epoch 11/30 |  Train Loss: 6.2942 |  Val Loss: 7.0333 |  Time: 0.46s
Epoch 12/30 |  Train Loss: 6.2063 |  Val Loss: 6.9888 |  Time: 0.45s
Epoch 13/30 |  Train Loss: 6.1284 |  Val Loss: 6.9343 |  Time: 0.45s
Epoch 14/30 |  Train Loss: 6.0535 |  Val Loss: 6.8931 |  Time: 0.51s
Epoch 15/30 |  Train Loss: 5.9884 |  Val 

## Experiment 1 Analysis

Training loss decreased steadily, showing that the model learned from the small dataset. Validation loss also improved at first, but eventually plateaued, suggesting limited generalization from only 100 examples.

In [19]:
train_dataset_1000 = dataset["train"].select(range(1000))
val_dataset_1000 = dataset["test"].select(range(100))

tokenized_dataset_1000 = train_dataset_1000.map(preprocess)
tokenized_dataset_1000.set_format(type="torch")

val_tokenized_1000 = val_dataset_1000.map(preprocess)
val_tokenized_1000.set_format(type="torch")

from torch.utils.data import DataLoader

dataloader_1000 = DataLoader(tokenized_dataset_1000, batch_size=8, shuffle=True)
val_dataloader_1000 = DataLoader(val_tokenized_1000, batch_size=8, shuffle=False)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [20]:
model_1000 = CustomSeq2SeqTransformer(
    no_enc_layers=2,
    no_dec_layers=2,
    emb_size=128,
    no_head=4,
    src_vcb_size=len(tokenizer),
    tgt_vcb_size=len(tokenizer),
    dim_feedforward=512,
    dropout=0.1
)

## Experiment 2: Scaling to 1000 Training Examples

After verifying the pipeline on 100 examples, the model is trained on 1000 examples. This tests whether increasing the amount of training data improves validation performance.

In [21]:
import time
import torch
import torch.nn as nn
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_1000 = model_1000.to(device)

loss_fn_1000 = nn.CrossEntropyLoss(ignore_index=pad_token_id)
optimizer_1000 = torch.optim.AdamW(model_1000.parameters(), lr=3e-4)

no_epochs_1000 = 30
best_vloss_1000 = 1_000_000.
model_transformer_path_1000 = ""
start_time_1000 = time.time()

pad_token_id = tokenizer.pad_token_id

for epoch in range(no_epochs_1000):
    running_loss_1000 = 0.0
    epoch_start_time_1000 = time.time()
    # Here, we use enumerate(training_loader) instead of
    # iter(training_loader) so that we can track the batch
    # index and do some intra-epoch reporting
    #We initialize the model to training mode
    model_1000.train()
    for i, data in enumerate(dataloader_1000):
      src = data["input_ids"].to(device)
      labels = data["labels"].to(device)

      decoder_input = labels[:, :-1]
      decoder_target = labels[:, 1:]


      src_padding_mask = src == pad_token_id
      tgt_padding_mask = decoder_input == pad_token_id
      memory_key_padding_mask = src_padding_mask

      tgt_mask = generate_subsequent_mask(decoder_input.size(1)).to(device)

      optimizer_1000.zero_grad()

      output = model_1000( src, decoder_input, src_mask=None, trg_mask=tgt_mask, src_padding_mask=src_padding_mask,
                     tgt_padding_mask=tgt_padding_mask, memory_key_padding_mask=memory_key_padding_mask)
      vocab_size = output.shape[-1]
      loss = loss_fn_1000(output.reshape(-1, vocab_size), decoder_target.reshape(-1))

      loss.backward()
      torch.nn.utils.clip_grad_norm_(model_1000.parameters(), max_norm=1.0)
      optimizer_1000.step()

      running_loss_1000 += loss.item()

    model_1000.eval()
    val_loss_1000 = 0.0
    with torch.no_grad():
      for i, val_data in enumerate(val_dataloader_1000):
        src = val_data["input_ids"].to(device)
        labels = val_data["labels"].to(device)

        decoder_input = labels[:, :-1]
        decoder_target = labels[:, 1:]


        src_padding_mask = src == pad_token_id
        tgt_padding_mask = decoder_input == pad_token_id
        memory_key_padding_mask = src_padding_mask

        tgt_mask = generate_subsequent_mask(decoder_input.size(1)).to(device)
        output = model_1000(src,decoder_input,src_mask=None,trg_mask=tgt_mask,
                         src_padding_mask=src_padding_mask,tgt_padding_mask=tgt_padding_mask,
                         memory_key_padding_mask=memory_key_padding_mask)

        vocab_size = output.shape[-1]

        loss = loss_fn_1000(output.reshape(-1, vocab_size),decoder_target.reshape(-1))

        val_loss_1000 += loss.item()


    avg_train_loss_1000 = running_loss_1000 / len(dataloader_1000)
    avg_val_loss_1000 = val_loss_1000 / len(val_dataloader_1000)
    if avg_val_loss_1000 < best_vloss_1000:
      best_vloss_1000 = avg_val_loss_1000
      torch.save(model_1000.state_dict(), "best_custom_transformer_1000.pt")
    epoch_end_1000 = time.time()

    print( f"Epoch {epoch+1}/{no_epochs_1000} | ", f"Train Loss: {avg_train_loss_1000:.4f} | ", f"Val Loss: {avg_val_loss_1000:.4f} | ", f"Time: {epoch_end_1000 - epoch_start_time_1000:.2f}s")

end_time_1000 = time.time()
print("Finished training. Time:", end_time_1000 - start_time_1000, "seconds")

Epoch 1/30 |  Train Loss: 8.2452 |  Val Loss: 6.8477 |  Time: 2.62s
Epoch 2/30 |  Train Loss: 6.5559 |  Val Loss: 6.3844 |  Time: 2.63s
Epoch 3/30 |  Train Loss: 6.1706 |  Val Loss: 6.1084 |  Time: 2.60s
Epoch 4/30 |  Train Loss: 5.9022 |  Val Loss: 5.9116 |  Time: 2.59s
Epoch 5/30 |  Train Loss: 5.6881 |  Val Loss: 5.7638 |  Time: 2.60s
Epoch 6/30 |  Train Loss: 5.5160 |  Val Loss: 5.6536 |  Time: 2.58s
Epoch 7/30 |  Train Loss: 5.3642 |  Val Loss: 5.5632 |  Time: 2.79s
Epoch 8/30 |  Train Loss: 5.2315 |  Val Loss: 5.4859 |  Time: 2.84s
Epoch 9/30 |  Train Loss: 5.0994 |  Val Loss: 5.4281 |  Time: 2.61s
Epoch 10/30 |  Train Loss: 4.9782 |  Val Loss: 5.3752 |  Time: 2.59s
Epoch 11/30 |  Train Loss: 4.8620 |  Val Loss: 5.3399 |  Time: 2.66s
Epoch 12/30 |  Train Loss: 4.7531 |  Val Loss: 5.2976 |  Time: 2.61s
Epoch 13/30 |  Train Loss: 4.6460 |  Val Loss: 5.2583 |  Time: 2.61s
Epoch 14/30 |  Train Loss: 4.5447 |  Val Loss: 5.2420 |  Time: 2.63s
Epoch 15/30 |  Train Loss: 4.4505 |  Val Lo

## Experiment 2 Analysis

With 1000 examples, validation loss improved more than in the 100-example experiment. However, after around epoch 20, validation loss flattened while training loss continued to decrease, indicating mild overfitting.

In [22]:
train_dataset_full = dataset["train"]
val_dataset_full = dataset["test"].select(range(300))

tokenized_dataset_full = train_dataset_full.map(preprocess)
tokenized_dataset_full.set_format(type="torch")

val_tokenized_full = val_dataset_full.map(preprocess)
val_tokenized_full.set_format(type="torch")

dataloader_full = DataLoader(tokenized_dataset_full, batch_size=8, shuffle=True)
val_dataloader_full = DataLoader(val_tokenized_full, batch_size=8, shuffle=False)

Map:   0%|          | 0/18949 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [23]:
model_full = CustomSeq2SeqTransformer(
    no_enc_layers=2,
    no_dec_layers=2,
    emb_size=128,
    no_head=4,
    src_vcb_size=len(tokenizer),
    tgt_vcb_size=len(tokenizer),
    dim_feedforward=512,
    dropout=0.1
)

## Experiment 3: Full BillSum Training Set

The final custom Transformer experiment uses the full BillSum training split. A 300-example validation subset is used during training to monitor generalization while keeping validation time manageable.

In [24]:
import time
import torch
import torch.nn as nn
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_full = model_full.to(device)

loss_fn_full = nn.CrossEntropyLoss(ignore_index=pad_token_id)
optimizer_full = torch.optim.AdamW(model_full.parameters(), lr=3e-4)

no_epochs_full = 30
best_vloss_full = 1_000_000.
model_transformer_path_full = ""
start_time_full = time.time()

pad_token_id = tokenizer.pad_token_id

for epoch in range(no_epochs_full):
    running_loss_full = 0.0
    epoch_start_time_full = time.time()
    # Here, we use enumerate(training_loader) instead of
    # iter(training_loader) so that we can track the batch
    # index and do some intra-epoch reporting
    #We initialize the model to training mode
    model_full.train()
    for i, data in enumerate(dataloader_full):
      src = data["input_ids"].to(device)
      labels = data["labels"].to(device)

      decoder_input = labels[:, :-1]
      decoder_target = labels[:, 1:]


      src_padding_mask = src == pad_token_id
      tgt_padding_mask = decoder_input == pad_token_id
      memory_key_padding_mask = src_padding_mask

      tgt_mask = generate_subsequent_mask(decoder_input.size(1)).to(device)

      optimizer_full.zero_grad()

      output = model_full( src, decoder_input, src_mask=None, trg_mask=tgt_mask, src_padding_mask=src_padding_mask,
                     tgt_padding_mask=tgt_padding_mask, memory_key_padding_mask=memory_key_padding_mask)
      vocab_size = output.shape[-1]
      loss = loss_fn_full(output.reshape(-1, vocab_size), decoder_target.reshape(-1))

      loss.backward()
      torch.nn.utils.clip_grad_norm_(model_full.parameters(), max_norm=1.0)
      optimizer_full.step()

      running_loss_full += loss.item()

    model_full.eval()
    val_loss_full = 0.0
    with torch.no_grad():
      for i, val_data in enumerate(val_dataloader_full):
        src = val_data["input_ids"].to(device)
        labels = val_data["labels"].to(device)

        decoder_input = labels[:, :-1]
        decoder_target = labels[:, 1:]


        src_padding_mask = src == pad_token_id
        tgt_padding_mask = decoder_input == pad_token_id
        memory_key_padding_mask = src_padding_mask

        tgt_mask = generate_subsequent_mask(decoder_input.size(1)).to(device)
        output = model_full(src,decoder_input,src_mask=None,trg_mask=tgt_mask,
                         src_padding_mask=src_padding_mask,tgt_padding_mask=tgt_padding_mask,
                         memory_key_padding_mask=memory_key_padding_mask)

        vocab_size = output.shape[-1]

        loss = loss_fn_full(output.reshape(-1, vocab_size),decoder_target.reshape(-1))

        val_loss_full += loss.item()


    avg_train_loss_full = running_loss_full / len(dataloader_full)
    avg_val_loss_full = val_loss_full / len(val_dataloader_full)
    if avg_val_loss_full < best_vloss_full:
      best_vloss_full = avg_val_loss_full
      torch.save(model_full.state_dict(), "best_custom_transformer_full.pt")
    epoch_end_full = time.time()

    print( f"Epoch {epoch+1}/{no_epochs_full} | ", f"Train Loss: {avg_train_loss_full:.4f} | ", f"Val Loss: {avg_val_loss_full:.4f} | ", f"Time: {epoch_end_full - epoch_start_time_full:.2f}s")

end_time_full = time.time()
print("Finished training. Time:", end_time_full - start_time_full, "seconds")

Epoch 1/30 |  Train Loss: 5.6895 |  Val Loss: 4.8686 |  Time: 45.01s
Epoch 2/30 |  Train Loss: 4.7412 |  Val Loss: 4.4699 |  Time: 45.01s
Epoch 3/30 |  Train Loss: 4.4095 |  Val Loss: 4.2623 |  Time: 44.94s
Epoch 4/30 |  Train Loss: 4.1960 |  Val Loss: 4.1313 |  Time: 45.03s
Epoch 5/30 |  Train Loss: 4.0394 |  Val Loss: 4.0507 |  Time: 45.22s
Epoch 6/30 |  Train Loss: 3.9222 |  Val Loss: 3.9738 |  Time: 45.04s
Epoch 7/30 |  Train Loss: 3.8268 |  Val Loss: 3.9375 |  Time: 45.05s
Epoch 8/30 |  Train Loss: 3.7463 |  Val Loss: 3.8800 |  Time: 44.97s
Epoch 9/30 |  Train Loss: 3.6775 |  Val Loss: 3.8677 |  Time: 45.31s
Epoch 10/30 |  Train Loss: 3.6187 |  Val Loss: 3.8295 |  Time: 45.17s
Epoch 11/30 |  Train Loss: 3.5659 |  Val Loss: 3.7979 |  Time: 45.04s
Epoch 12/30 |  Train Loss: 3.5205 |  Val Loss: 3.7804 |  Time: 45.23s
Epoch 13/30 |  Train Loss: 3.4768 |  Val Loss: 3.7642 |  Time: 46.62s
Epoch 14/30 |  Train Loss: 3.4387 |  Val Loss: 3.7422 |  Time: 45.30s
Epoch 15/30 |  Train Loss: 3.

## Experiment 3 Analysis

Training on the full BillSum dataset significantly improved validation loss. Unlike the smaller experiments, validation loss continued improving through later epochs, suggesting that the larger dataset helped reduce overfitting and improve generalization.

## Greedy Decoding for Summary Generation

Since this custom model does not have Hugging Face’s built-in.generate() method, a manual greedy decoding function is implemented.

The process is:
* Encode the bill text once.
* Start the decoder with the `<bos>` token.
* Predict the most likely next token.
* Append that token to the generated sequence.
* Repeat until `<eos>` or maximum length is reached.

A small repetition penalty is applied to reduce repeated phrases.

In [25]:
def generate_summary(model, src, max_len=128):
    model.eval()

    src = src.to(device)
    # Create a padding mask (true where padding exists)
    src_padding_mask = src == pad_token_id

    #Convert tokens to embeddings then add positional encodings
    # memory = representation of the input sequence

    memory = model.transformer.encoder(
        model.positional_encoding(model.src_embedding(src) * math.sqrt(model.emb_size)),
        src_key_padding_mask=src_padding_mask
    )
    # Start with BOS (beginning of sequence) token

    ys = torch.ones((1, 1), dtype=torch.long).fill_(tokenizer.bos_token_id).to(device)

    for _ in range(max_len):
        # Create causal mask so decoder cannot see future tokens
        tgt_mask = generate_subsequent_mask(ys.size(1)).to(device)
        # Pass through decoder (uses encoder memory)
        out = model.transformer.decoder(
            model.positional_encoding(model.trgt_embedding(ys) * math.sqrt(model.emb_size)),
            memory,
            tgt_mask=tgt_mask,
            memory_key_padding_mask=src_padding_mask
        )

        out = model.feedforward(out[:, -1])
        # Pick the token with highest probability (greedy decoding)
        # Append predicted token to sequence
        out[:, ys[0, -5:]] -= 10.0
        next_token = out.argmax(dim=-1).item()

        ys = torch.cat([ys, torch.tensor([[next_token]], device=device)], dim=1)
        # Stop if EOS (end of sequence) token is generated
        if next_token == tokenizer.eos_token_id:
            break

    return ys

## Qualitative Evaluation

We compare the generated summary with the reference summary to inspect whether the model learned legal summarization patterns.

In [26]:
sample = next(iter(val_dataloader_full))
src_sample = sample["input_ids"][0].unsqueeze(0)

tokens = generate_summary(model_full, src_sample, max_len=40)
print("REFERENCE:")
print(tokenizer.decode(sample["labels"][0], skip_special_tokens=True))
print("\nGENERATED:")
print(tokenizer.decode(tokens[0], skip_special_tokens=True))

REFERENCE:
Amends the Water Resources Development Act of 1999 to: (1) authorize appropriations for FY 1999 through 2009 for implementation of a long-term resource monitoring program with respect to the Upper Mississippi River Environmental Management Program (currently, such funding is designated for a program for the planning, construction, and evaluation of measures for fish and wildlife habitat rehabilitation and enhancement); (2) authorize the Secretary of the Army to carry out modifications to the navigation project for the Delaware River, Pennsylvania and Delaware, if such project as modified is technically sound, environmentally (currently, economically) acceptable, and economically justified; (3) subject certain previously deauthorized water

GENERATED:
Amends the Marine Sanctuaries of the Interior to authorize appropriations for the Secretary of Agriculture to make grants for the operation of such grants to States for the operation of such projects.
Authorizes appropriations


## Qualitative Analysis

The custom Transformer learned domain-specific legal phrasing and often begins summaries in a realistic legislative style. However, the generated text may repeat phrases, which is common in greedy decoding for models trained from scratch.

## Quantitative Evaluation with ROUGE

ROUGE measures overlap between generated summaries and reference summaries.

* ROUGE-1 measures unigram overlap.
* ROUGE-2 measures bigram overlap.
* ROUGE-L measures longest common subsequence overlap.

The model is evaluated on 30 validation examples.

In [27]:
import evaluate

rouge = evaluate.load("rouge")

predictions = []
references = []

num_samples = 30  # pick 20–50
count = 0

model_full.eval()

with torch.no_grad():
    for batch in val_dataloader_full:
        src_batch = batch["input_ids"].to(device)
        labels_batch = batch["labels"]

        for i in range(len(src_batch)):
            # generate summary
            tokens = generate_summary(model_full, src_batch[i].unsqueeze(0), max_len=60)
            pred_text = tokenizer.decode(tokens[0], skip_special_tokens=True)

            # reference summary
            ref_text = tokenizer.decode(labels_batch[i], skip_special_tokens=True)

            predictions.append(pred_text)
            references.append(ref_text)

            count += 1
            if count >= num_samples:
                break

        if count >= num_samples:
            break

# compute average ROUGE
scores = rouge.compute(predictions=predictions, references=references)

print(scores)

{'rouge1': np.float64(0.34706269943151613), 'rouge2': np.float64(0.12920929863058656), 'rougeL': np.float64(0.26421752135995846), 'rougeLsum': np.float64(0.280441386229797)}


## ROUGE Analysis

The custom Transformer achieved measurable ROUGE performance, showing that it learned some summarization behavior. However, performance is expected to be lower than pretrained BART because this model was trained from scratch and does not benefit from large-scale language pretraining.

# Conclusion

This project implemented a custom encoder-decoder Transformer for legal bill summarization using the BillSum dataset. The experiments showed that the model successfully learned from the data, with validation loss improving as the training set increased from 100 examples to the full training split.

The custom Transformer was able to generate summaries with legal-domain phrasing, but it also showed limitations such as repetition during greedy decoding. This highlights why pretrained models such as BART are typically stronger for summarization: they already contain language knowledge learned from large-scale pretraining.

Future work could compare this custom model directly against pretrained BART.

### References
* For tokenization
https://huggingface.co/docs/transformers/tasks/summarization
* For Transformer Architecture
https://github.com/pytorch/examples/blob/main/language_translation/src/model.py